# Sample And Upload Validation

Sampling happens locally. This notebook verifies that the expected sampled CSV files were uploaded to the raw Unity Catalog volume before bronze ingestion begins.

## Run Inputs
- `catalog`: optional override for the active catalog
- `schema`: schema that holds the RetailPulse volumes
- `raw_volume`: volume that should contain the sampled CSV files

## Source Inputs
- Uploaded sample files in `/Volumes/<catalog>/<schema>/<raw_volume>/`
- Required filenames:
  - `orders.csv`
  - `order_products__prior.csv`
  - `order_products__train.csv`
  - `products.csv`
  - `aisles.csv`
  - `departments.csv`

## Outputs
- A presence-check table for every required file
- A lightweight row-count preview for each CSV
- A hard failure if any required upload is missing

## Workflow
1. Resolve the raw volume path from widgets.
2. Confirm every required file is present.
3. Preview row counts so operators can catch empty uploads early.



## Capture Runtime Inputs

The notebook stays read-only. Its purpose is to gate the bronze step, not to mutate the uploaded files.



In [ ]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "retailpulse")
dbutils.widgets.text("raw_volume", "retailpulse_raw")



## Resolve The Raw Volume Path

This path becomes the single source of truth for the upload checkpoint used by the bronze ingest notebook.



In [ ]:
catalog = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
schema = dbutils.widgets.get("schema") or spark.sql("SELECT current_schema()").first()[0]
raw_volume = dbutils.widgets.get("raw_volume")
raw_root = f"/Volumes/{catalog}/{schema}/{raw_volume}"

required_files = [
    "orders.csv",
    "order_products__prior.csv",
    "order_products__train.csv",
    "products.csv",
    "aisles.csv",
    "departments.csv",
]



## Validate Required Uploads

Treat the volume listing as the gate before reading any CSV content. This fails fast when a file is missing or uploaded under the wrong name.



In [ ]:
uploaded = {item.name.rstrip("/") for item in dbutils.fs.ls(raw_root)}
status_rows = [(name, name in uploaded, f"{raw_root}/{name}") for name in required_files]
display(spark.createDataFrame(status_rows, ["file_name", "present", "path"]))

missing = [name for name in required_files if name not in uploaded]
assert not missing, f"Missing uploaded files: {', '.join(missing)}"



## Preview Row Counts

These counts are not a full contract test, but they are enough to catch empty or obviously wrong uploads before bronze tables are overwritten.



In [ ]:
preview_counts = []
for name in required_files:
    count = (
        spark.read.option("header", True)
        .csv(f"{raw_root}/{name}")
        .count()
    )
    preview_counts.append((name, count))

display(spark.createDataFrame(preview_counts, ["file_name", "row_count"]))



## Local Sampling Command

If you need to rebuild the sampled files locally, use the deterministic sampler before re-uploading:

```powershell
python scripts/sample_instacart.py `
  --input-dir C:\path\to\instacart_raw `
  --output-dir C:\path\to\retailpulse_sample
```

Once the required files are present and non-empty, continue with `02_bronze_ingest`.
